[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/ligo/ligo.ipynb)

# Gravitational-Wave Antenna Patterns

This example computes how strongly an L-shaped gravitational-wave interferometer such as LIGO responds to a wave arriving from each direction on the sky. Rather than pointing at one target, a ground-based interferometer hears waves from nearly every direction at once with directional sensitivity set by the geometry of its two perpendicular arms. Comparing the Hanford and Livingston sites across the celestial sphere shows how a nearly anti-aligned pair covers the sky, while sweeping the polarization angle demonstrates that gravitational radiation is a spin-two field.

In spacetime algebra, a transverse gravitational wave is modelled as a symmetric traceless duality map `Strain = Plane+ <- Point-` orthogonal to both the observer's time axis and the wave's arrival direction. Pairing an arm displacement with its strain plane `(strain(arm) & arm) / 2` gives the arm's fractional length change directly without dualizing, while the plus and cross basis strains `east * (east & Point) - north * (north & Point)` and `east * (north & Point) + north * (east & Point)` transform under spatial rotors `frames >> y` and `frames >> z`. One spatial rotor family places both interferometer sites on Earth and arrival directions on the sky, and spacetime metric duality `overhead_plus(rest_ring).dual()` enters only when converting tidal reading planes into spatial shifts of a ring of free test masses.

In the $3 \times 3$ Cartesian tensor notation of gravitational-wave astronomy, the arrival direction $(\theta, \phi)$ and polarization angle $\psi$ define transverse unit vectors $\hat{\mathbf m}, \hat{\mathbf n}$ rotated from spherical basis vectors $(\hat{\boldsymbol\theta}, \hat{\boldsymbol\phi})$ by $\psi$, building polarization tensors $e^+_{ij} = m_i m_j - n_i n_j$ and $e^\times_{ij} = m_i n_j + n_i m_j$ contracted against the detector tensor $D^{ij} = \tfrac12(u^i u^j - v^i v^j)$ as $F_+ = D^{ij} e^+_{ij}$ and $F_\times = D^{ij} e^\times_{ij}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from examples.relativity.ligo import render
from examples.relativity.ligo.scenarios import (
    arm_azimuths,
    detector_names,
    frame_duration_ms,
    n_frames,
    n_masses,
    n_sky_samples,
    polarization_size,
    site_latitudes,
    site_longitudes,
    strain_scale,
)
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
# Spatial displacements and sky directions lie orthogonal to `mv.t` as intrinsic 4-vectors.
Point = ga.gatype.antivector()                             # Antivector-
Plane = ga.gatype.vector()                                 # Vector+
Rotor = ga.gatype.rotor()                                  # Rotor+
# Transverse tidal metric perturbation mapping spatial displacements to spatial reading planes.
Strain = ga.gatype((Plane, Point))                         # Plane+ <- Point-
first_axis = mv.x.dual()                                   # Antivector-

## Shared rotor frames for Earth sites and sky directions

Turning the first spatial axis to a given latitude and longitude simultaneously carries the second spatial axis to local east and the third to local north. At an observatory site those two planes bound the horizontal plane containing the interferometer arms; for a point on the celestial sphere they span the transverse wavefront of radiation arriving from straight overhead that point on Earth.

In [ ]:
def earth_frame(latitudes: np.ndarray, longitudes: np.ndarray) -> Rotor:
    """Spatial rotors (`[...] Rotor+`) turn the first axis to each `(latitudes, longitudes)` direction in radians and align the second and third axes with local east and north."""
    latitude_turn = (mv.xz * (-latitudes / 2)).exp()                            # [...] Rotor+
    longitude_turn = (mv.xy * (-longitudes / 2)).exp()                          # [...] Rotor+
    return longitude_turn * latitude_turn


def detector_arms(sites: Rotor, azimuths: np.ndarray) -> Point:
    """Unit spatial arm displacements (`[..., 2] Point-`) point along `azimuths` (`[..., 2]`, radians clockwise from north) in each site's horizontal plane."""
    east, north = sites[..., None] >> mv.y, sites[..., None] >> mv.z            # [..., 1] Plane+, Plane+
    azimuth_turn = ((north ^ east) * (-azimuths / 2)).exp()                     # [..., 2] Rotor+
    return azimuth_turn >> north.dual()                                         # [..., 2] Point-


sites = earth_frame(np.radians(site_latitudes), np.radians(site_longitudes))    # [2] Rotor+
arms = detector_arms(sites, np.radians(arm_azimuths))                           # [2, 2] Point-
latitudes = np.radians(np.linspace(-90, 90, n_sky_samples))
longitudes = np.radians(np.linspace(-180, 180, 2 * n_sky_samples - 1))
sky_frames = earth_frame(latitudes[:, None], longitudes)                        # [n_sky_samples, 2 * n_sky_samples - 1] Rotor+
directions = sky_frames >> first_axis                                           # [n_sky_samples, 2 * n_sky_samples - 1] Point-
zeniths = sites >> first_axis                                                   # [2] Point-
arms

## Antenna patterns and two-detector network alignment

For each arrival frame, the plus and cross basis strains stretch and shear space across local east and north while leaving the propagation axis untouched. Subtracting the fractional stretches `(strain(arm) & arm) / 2` of the two arms gives the detector's antenna patterns, and their root sum of squares measures directional sensitivity independently of how the wave's transverse axes are oriented.

In [ ]:
def wave_strains(frames: Rotor) -> tuple[Strain, Strain]:
    """Plus and cross tidal strain maps (`[...] Plane+ <- Point-`) act across the transverse east and north planes of `frames`."""
    east, north = frames >> mv.y, frames >> mv.z                                # [...] Plane+, Plane+
    plus = east * (east & Point) - north * (north & Point)                      # [...] Plane+ <- Point-
    cross = east * (north & Point) + north * (east & Point)                     # [...] Plane+ <- Point-
    return plus, cross


def arm_response(arms: Point, strain: Strain) -> Scalar:
    """Differential arm stretch (`[...] Scalar+`) subtracts the two arm readings `(strain(arm) & arm) / 2`."""
    stretches = (strain[..., None](arms) & arms) / 2                            # [..., 2] Scalar+
    return stretches[..., 0] - stretches[..., 1]                                # [...] Scalar+


plus, cross = wave_strains(sky_frames)                                          # [n_sky_samples, 2 * n_sky_samples - 1] Plane+ <- Point-, Plane+ <- Point-
plus_response = arm_response(arms[:, None, None], plus)                         # [2, n_sky_samples, 2 * n_sky_samples - 1] Scalar+
cross_response = arm_response(arms[:, None, None], cross)                       # [2, n_sky_samples, 2 * n_sky_samples - 1] Scalar+

In [ ]:
def directional_sensitivity(plus_response: Scalar, cross_response: Scalar) -> Scalar:
    """Root-sum-of-squares antenna sensitivity (`[...] Scalar+`) combines `plus_response` and `cross_response` invariantly under polarization turns."""
    return (plus_response.squared() + cross_response.squared()).square_root()


sensitivities = directional_sensitivity(plus_response, cross_response)          # [2, n_sky_samples, 2 * n_sky_samples - 1] Scalar+
network_sensitivity = sensitivities.squared().sum(axis=0).square_root()         # [n_sky_samples, 2 * n_sky_samples - 1] Scalar+
# Cosine alignment between the two detectors' polarization response pairs: -1 where their signs oppose.
alignment = (plus_response[0] * plus_response[1] + cross_response[0] * cross_response[1]) / (
    sensitivities[0] * sensitivities[1]
)                                                                               # [n_sky_samples, 2 * n_sky_samples - 1] Scalar+
save_figure(render.patterns_figure(directions, plus_response, cross_response, sensitivities, zeniths, detector_names), "ligo_patterns");

Each row maps one observatory in Mollweide projection, placing every arrival direction over its sub-zenith point on Earth (marked by the cyan disc): red and blue lobes show positive and negative differential arm stretch for plus and cross waves, while the third column shows polarization-averaged sensitivity peaking bright yellow overhead and underfoot and dropping to black at the four horizontal arm-bisector blind spots.

In [ ]:
save_figure(render.network_figure(directions, network_sensitivity, alignment, zeniths, detector_names), "ligo_network");

On the left, combined Hanford–Livingston sensitivity stays high over the same broad sky regions because the two interferometers lie nearly coplanar; on the right, deep blue covers more than ninety percent of the sky because their arms were built nearly anti-aligned so a passing wave registers with opposite sign at the two sites.

## Spin-two polarization sweep

Rotating the transverse wavefront axes about the propagation direction carries the plus strain into the cross strain after an eighth of a turn and into minus itself after a quarter turn, repeating after half a turn. Streaming a half-turn sweep at Hanford pairs the tidal deformation of an overhead ring of free test masses with the shifting sky response.

In [ ]:
def polarized_frame(frames: Rotor, angles: np.ndarray) -> Rotor:
    """Spatial rotors (`[...] Rotor+`) rotate `frames` about their propagation axis by `angles` in radians."""
    polarization_turn = (mv.yz * (-angles / 2)).exp()                           # [...] Rotor+
    return frames * polarization_turn


def polarization_states(
    site: Rotor,
    arms: Point,
    sky_frames: Rotor,
    angles: np.ndarray,
    rest_ring: Point,
    strain_scale: float,
) -> Iterator[tuple[Scalar, Point]]:
    """Yields `(sky_response, deformed_ring)` pairs (`[n_latitudes, n_longitudes] Scalar+`, `[n_masses] Point-`) as the wave polarization steps through `angles`."""
    for angle in angles:
        turned_sky = polarized_frame(sky_frames, np.asarray(angle))             # [n_sky_samples, 2 * n_sky_samples - 1] Rotor+
        plus_sky, _ = wave_strains(turned_sky)                                  # [n_sky_samples, 2 * n_sky_samples - 1] Plane+ <- Point-
        sky_response = arm_response(arms, plus_sky)                             # [n_sky_samples, 2 * n_sky_samples - 1] Scalar+
        turned_overhead = polarized_frame(site, np.asarray(angle))              # [] Rotor+
        overhead_plus, _ = wave_strains(turned_overhead)                        # [] Plane+ <- Point-
        deformed_ring = rest_ring + strain_scale * overhead_plus(rest_ring).dual()   # [n_masses] Point-
        yield sky_response, deformed_ring


east, north = sites[0] >> mv.y, sites[0] >> mv.z                                # [] Plane+, Plane+
ring_angles = np.linspace(0, 2 * np.pi, n_masses, endpoint=False)
ring_rotors = ((east ^ north) * (-ring_angles / 2)).exp()                       # [n_masses] Rotor+
rest_ring = ring_rotors >> east.dual()                                          # [n_masses] Point-
angles = np.linspace(0, np.pi, n_frames, endpoint=False)
streamed_states = show_states(
    polarization_states(sites[0], arms[0], sky_frames, angles, rest_ring, strain_scale),
    render.polarization_animation,
    directions,
    sites[0],
    arms[0],
    detector_names[0],
    polarization_size,
    name="ligo_polarization",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

On the left, the blue ring of free test masses stretches into an ellipse aligned with the polarization axes relative to Hanford's orange arms (grey circle at rest); on the right, the red and blue sky response bands complete a full cycle as the ellipse rotates through half a turn.

In [ ]:
# checks
# Sensitivity equals 1 straight overhead each site and never exceeds 1 anywhere on the sky.
overhead_plus, overhead_cross = wave_strains(sites)                             # [2] Plane+ <- Point-, Plane+ <- Point-
overhead_sensitivity = directional_sensitivity(
    arm_response(arms, overhead_plus),
    arm_response(arms, overhead_cross),
)                                                                               # [2] Scalar+
assert ((overhead_sensitivity - 1).abs() < 1e-10).all()
assert (sensitivities <= 1 + 1e-10).all()

# Waves arriving along the horizontal arm bisector vanish in both polarizations.
bisector = arms[:, 0] + arms[:, 1]
bisector = bisector / bisector.antinorm()                                       # [2] Point-
bisector_frame = (1 + bisector * first_axis).normalized()                       # [2] Rotor+
blind_plus, blind_cross = wave_strains(bisector_frame)                          # [2] Plane+ <- Point-, Plane+ <- Point-
blind_sensitivity = directional_sensitivity(
    arm_response(arms, blind_plus),
    arm_response(arms, blind_cross),
)                                                                               # [2] Scalar+
assert (blind_sensitivity.abs() < 1e-10).all()

# Spin-two law: rotating transverse axes by `angles` mixes plus and cross responses by `2 * angles`.
swept_responses = stack([state[0] for state in streamed_states])                # [n_frames, n_sky_samples, 2 * n_sky_samples - 1] Scalar+
mixed = (
    plus_response[0] * np.cos(2 * angles)[:, None, None]
    + cross_response[0] * np.sin(2 * angles)[:, None, None]
)                                                                               # [n_frames, n_sky_samples, 2 * n_sky_samples - 1] Scalar+
assert ((swept_responses - mixed).abs() < 1e-10).all()

# Hanford and Livingston read opposite signs across more than 90% of the sky.
assert np.mean(alignment < -0.5) > 0.9